In [1]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torch
from tqdm import tqdm

In [2]:
#Set dataset, model and tokenizer
dataset = load_dataset('glue', 'mrpc')
model = AutoModelForSequenceClassification.from_pretrained('bert-base-cased')
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

Found cached dataset glue (/home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


  0%|          | 0/3 [00:00<?, ?it/s]

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPretraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at b

In [3]:
# Define the training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

In [4]:
#Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["sentence1"], examples["sentence2"], truncation=True, padding="max_length")


def reformat_dataset(dataset):
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset = tokenized_dataset.map(lambda rows: {'labels': rows['label']}, batched=True)
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
    return tokenized_dataset

tokenized_dataset = reformat_dataset(dataset)




Loading cached processed dataset at /home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad/cache-d2ffa505ac8a05f8.arrow
Loading cached processed dataset at /home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad/cache-48ba80a0dd761d30.arrow
Loading cached processed dataset at /home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad/cache-b1f8383f6aa6d074.arrow
Loading cached processed dataset at /home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad/cache-ac8ef27305e230a7.arrow
Loading cached processed dataset at /home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad/cache-5179515acb1361cc.arrow
Loading cached processed dataset at /home/ubuntu/.cache/huggingfa

In [5]:
# Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
)


In [6]:

# Train the model
trainer.train()

Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

Iteration:   0%|          | 0/115 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/13 [00:00<?, ?it/s]

{'eval_loss': 0.5015673573110618, 'epoch': 1.0, 'total_flos': 1220467873013760, 'step': 115}


Iteration:   0%|          | 0/115 [00:00<?, ?it/s]

...

{'eval_loss': 0.43084392331394494, 'epoch': 2.0, 'total_flos': 2440935746027520, 'step': 230}


Iteration:   0%|          | 0/115 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/13 [00:00<?, ?it/s]

{'eval_loss': 0.35840479240697976, 'epoch': 3.0, 'total_flos': 3661403619041280, 'step': 345}


TrainOutput(global_step=345, training_loss=0.43673277039458785)

In [8]:
torch.save(model.state_dict(), 'state.pt')
torch.save(model, 'model.pt')